In [22]:
import pandas as pd

df = pd.read_csv('recipe.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4172 entries, 0 to 4171
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   recipe_id             4172 non-null   int64  
 1   title                 4172 non-null   object 
 2   instructions          4172 non-null   object 
 3   image                 4172 non-null   object 
 4   cooking_time          4172 non-null   int64  
 5   slug                  4172 non-null   object 
 6   measured_ingredients  4172 non-null   object 
 7   calories              4172 non-null   float64
 8   protein               4172 non-null   float64
 9   fat                   4172 non-null   float64
 10  carbohydrate          4172 non-null   float64
 11  sodium                4172 non-null   float64
 12  rating                4172 non-null   float64
dtypes: float64(6), int64(2), object(5)
memory usage: 423.8+ KB


In [16]:
import pandas as pd

CSV_IN = "recipe.csv"
CSV_OUT = "recipe_fixed.csv"

# recipe_id -> correct image value (NO extension)
updates = {
    714: "-bloody-mary-tomato-toast-with-celery-and-horseradish-56389813",
    903: "-lentils-with-cucumbers-chard-and-poached-egg-51260640",
    906: "-hazelnut-butter-and-coffee-meringues-51260360",
    968: "-halibut-confit-with-leeks-coriander-and-lemon-51252690",
    1674: "-candy-corn-frozen-citrus-cream-pops-368770",
    3231: "-like-a-caesar-235836",
}

df = pd.read_csv(CSV_IN, encoding="utf-8")

if "recipe_id" not in df.columns or "image" not in df.columns:
    raise ValueError(f"CSV must contain 'recipe_id' and 'image' columns. Found: {list(df.columns)}")

# Apply updates
mask = df["recipe_id"].isin(updates.keys())
df.loc[mask, "image"] = df.loc[mask, "recipe_id"].map(updates)

# Verify all requested ids were found
found_ids = set(df.loc[mask, "recipe_id"].tolist())
missing_ids = set(updates.keys()) - found_ids
if missing_ids:
    print("WARNING: These recipe_id values were not found in the CSV (no update applied):")
    for rid in sorted(missing_ids):
        print(f"- {rid}")

# Show changed rows
print("Updated rows:")
print(df.loc[df["recipe_id"].isin(updates.keys()), ["recipe_id", "title", "image"]].to_string(index=False))

df.to_csv(CSV_OUT, index=False, encoding="utf-8")
print(f"\nWrote updated CSV to: {CSV_OUT}")

Updated rows:
 recipe_id                                                  title                                                          image
       714 "Bloody Mary" Tomato Toast with Celery and Horseradish -bloody-mary-tomato-toast-with-celery-and-horseradish-56389813
       903         Lentils with Cucumbers, Chard, and Poached Egg         -lentils-with-cucumbers-chard-and-poached-egg-51260640
       906                   Hazelnut Butter and Coffee Meringues                 -hazelnut-butter-and-coffee-meringues-51260360
       968        Halibut Confit With Leeks, Coriander, and Lemon        -halibut-confit-with-leeks-coriander-and-lemon-51252690
      1674                  "Candy Corn" Frozen Citrus Cream Pops                    -candy-corn-frozen-citrus-cream-pops-368770
      3231                                        "Like a Caesar"                                          -like-a-caesar-235836

Wrote updated CSV to: recipe_fixed.csv


In [19]:
import csv
from pathlib import Path

CSV_PATH = Path("./recipe.csv")
IMAGES_DIR = Path("./food-images")
IMAGE_EXT = ".jpg"

DRY_RUN = True  # set to False to actually delete files


def read_csv_image_names(csv_path: Path) -> set[str]:
    names: set[str] = set()

    with csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames or "image" not in reader.fieldnames:
            raise ValueError(f"CSV must contain an 'image' column. Found: {reader.fieldnames}")

        for row_num, row in enumerate(reader, start=2):  # header is line 1
            val = (row.get("image") or "").strip()
            if not val:
                continue
            names.add(val)

    return names


def main():
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")
    if not IMAGES_DIR.exists():
        raise FileNotFoundError(f"Images folder not found: {IMAGES_DIR.resolve()}")

    csv_images = read_csv_image_names(CSV_PATH)

    # 1) Check missing images for recipes (print expected filename)
    missing_expected = []
    for image_base in sorted(csv_images):
        expected_filename = f"{image_base}{IMAGE_EXT}"
        expected_path = IMAGES_DIR / expected_filename
        if not expected_path.exists():
            missing_expected.append((expected_filename, expected_path))

    print("=== CSV -> missing image files (recipes without images) ===")
    if missing_expected:
        for expected_filename, expected_path in missing_expected:
            print(f"MISSING: expected filename: {expected_filename} | expected path: {expected_path}")
        print(f"Total missing: {len(missing_expected)}")
    else:
        print("All recipes have corresponding images.")

    # 2) Remove extra images not referenced by CSV
    extras = []
    for jpg in IMAGES_DIR.glob(f"*{IMAGE_EXT}"):
        if jpg.stem not in csv_images:
            extras.append(jpg)

    print("\n=== Images -> extra files not referenced in CSV ===")
    if extras:
        for p in sorted(extras):
            print(f"EXTRA: {p.name} | path: {p}")
        print(f"Total extra: {len(extras)}")
    else:
        print("No extra images found.")

    if extras:
        if DRY_RUN:
            print("\nDRY_RUN=True, so nothing was deleted.")
            print("Set DRY_RUN=False to actually delete the EXTRA files above.")
        else:
            for p in extras:
                p.unlink()
            print(f"\nDeleted {len(extras)} extra image(s).")


if __name__ == "__main__":
    main()

=== CSV -> missing image files (recipes without images) ===
All recipes have corresponding images.

=== Images -> extra files not referenced in CSV ===
No extra images found.


In [20]:
import csv
from pathlib import Path

CSV_PATH = Path("./recipe.csv")
IMAGES_DIR = Path("./food-images")
IMAGE_EXT = ".jpg"

DRY_RUN = False  # set to False to actually delete files


def read_csv_image_names(csv_path: Path) -> set[str]:
    names: set[str] = set()

    with csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames or "image" not in reader.fieldnames:
            raise ValueError(f"CSV must contain an 'image' column. Found: {reader.fieldnames}")

        for row_num, row in enumerate(reader, start=2):  # header is line 1
            val = (row.get("image") or "").strip()
            if not val:
                continue
            names.add(val)

    return names


def main():
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")
    if not IMAGES_DIR.exists():
        raise FileNotFoundError(f"Images folder not found: {IMAGES_DIR.resolve()}")

    csv_images = read_csv_image_names(CSV_PATH)

    # 1) Check missing images for recipes (print expected filename)
    missing_expected = []
    for image_base in sorted(csv_images):
        expected_filename = f"{image_base}{IMAGE_EXT}"
        expected_path = IMAGES_DIR / expected_filename
        if not expected_path.exists():
            missing_expected.append((expected_filename, expected_path))

    print("=== CSV -> missing image files (recipes without images) ===")
    if missing_expected:
        for expected_filename, expected_path in missing_expected:
            print(f"MISSING: expected filename: {expected_filename} | expected path: {expected_path}")
        print(f"Total missing: {len(missing_expected)}")
    else:
        print("All recipes have corresponding images.")

    # 2) Remove extra images not referenced by CSV
    extras = []
    for jpg in IMAGES_DIR.glob(f"*{IMAGE_EXT}"):
        if jpg.stem not in csv_images:
            extras.append(jpg)

    print("\n=== Images -> extra files not referenced in CSV ===")
    if extras:
        for p in sorted(extras):
            print(f"EXTRA: {p.name} | path: {p}")
        print(f"Total extra: {len(extras)}")
    else:
        print("No extra images found.")

    if extras:
        if DRY_RUN:
            print("\nDRY_RUN=True, so nothing was deleted.")
            print("Set DRY_RUN=False to actually delete the EXTRA files above.")
        else:
            for p in extras:
                p.unlink()
            print(f"\nDeleted {len(extras)} extra image(s).")


if __name__ == "__main__":
    main()

=== CSV -> missing image files (recipes without images) ===
All recipes have corresponding images.

=== Images -> extra files not referenced in CSV ===
No extra images found.
